# Notebook 07 — Layer norm and residuals

Paired with `theory/07_layer_norm_residuals.md`. **Mode:** theory-first.

## QHMPC

**Question.** Verify five structural / empirical claims:
1. LN matches the PyTorch impl bit-for-bit on a hand example.
2. LN's two invariances (shift, scale) hold to fp precision.
3. Residual stacks preserve gradient flow at depth $L = 32$ where plain stacks vanish.
4. Pre-norm trains stably at depth 16; post-norm needs warmup or diverges.
5. Pre-norm residual stream norm grows monotonically with depth at init.

**Method.** numpy + torch. Experiments 4–5 train tiny stacks (~30 sec each on CPU).

**Prediction.** *Paste §7.7 predictions.*

**Confidence.** *[low/medium/high]*.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

np.random.seed(0); torch.manual_seed(0)

## Experiment 1 — LN by hand vs. PyTorch

In [ ]:
def ln_manual(x, eps=1e-5):
    mu = x.mean(-1, keepdims=True)
    var = ((x - mu) ** 2).mean(-1, keepdims=True)
    return (x - mu) / np.sqrt(var + eps)

x = np.array([1.0, 3.0, 5.0, 7.0])
manual = ln_manual(x)
torch_out = F.layer_norm(torch.tensor(x), normalized_shape=(4,)).numpy()
print(f'manual:  {manual}')
print(f'torch:   {torch_out}')
print(f'max abs diff: {np.max(np.abs(manual - torch_out)):.2e}')
print(f'manual mean / std: {manual.mean():.4e} / {manual.std():.4e}  (expect 0 / 1)')

**Sub-finding 1.** *Matched? Mean = 0, std ≈ 1?*

## Experiment 2 — LN invariances (Prop 7.2)

In [ ]:
x = np.random.randn(8)
shift = ln_manual(x + 100)
scale = ln_manual(5 * x)
base = ln_manual(x)
print(f'LN(x + 100) vs LN(x), max abs diff: {np.max(np.abs(shift - base)):.2e}')
print(f'LN(5x)      vs LN(x), max abs diff: {np.max(np.abs(scale - base)):.2e}')

**Sub-finding 2.** *Both invariances hold to fp precision?*

## Experiment 3 — gradient flow: residual vs. plain stack

Build $L = 32$ random Jacobian layers. With residuals, the gradient norm at input should remain $O(1)$. Without, it should vanish or explode.

In [ ]:
L_max = 64
d = 32
torch.manual_seed(0)
Js = [torch.randn(d, d) * (1.0 / np.sqrt(d)) for _ in range(L_max)]    # near-isometric per layer

def gradient_norm(L, residual):
    x = torch.randn(d, requires_grad=True)
    h = x
    for ell in range(L):
        f = Js[ell] @ h
        h = h + f if residual else f
    loss = h.sum()
    g = torch.autograd.grad(loss, x)[0]
    return g.norm().item()

Ls = [1, 2, 4, 8, 16, 32, 64]
with_res = [gradient_norm(L, residual=True) for L in Ls]
no_res  = [gradient_norm(L, residual=False) for L in Ls]

fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogy(Ls, with_res, 'o-', label='with residuals')
ax.semilogy(Ls, no_res,   's-', label='no residuals')
ax.set_xlabel('depth L'); ax.set_ylabel('||grad at input||')
ax.set_title('Gradient flow: residual vs. plain stack')
ax.legend(); plt.tight_layout(); plt.show()

**Sub-finding 3.** *Did the no-residuals stack vanish (or explode) at $L = 32$? Did residuals preserve $O(1)$ gradient norm?*

## Experiment 4 — pre-norm vs. post-norm trainability at depth

Tiny synthetic regression task. Train depth-16 stacks with both norm placements; compare loss curves.

In [ ]:
d = 32
n_train, n_steps = 256, 300
X = torch.randn(n_train, d)
W_target = torch.randn(d, d) / np.sqrt(d)
y = X @ W_target

class SubBlock(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.fc1 = nn.Linear(d, 4 * d)
        self.fc2 = nn.Linear(4 * d, d)
    def forward(self, x):
        return self.fc2(F.gelu(self.fc1(x)))

class Stack(nn.Module):
    def __init__(self, L, d, mode):
        super().__init__()
        self.mode = mode
        self.blocks = nn.ModuleList([SubBlock(d) for _ in range(L)])
        self.norms = nn.ModuleList([nn.LayerNorm(d) for _ in range(L)])
        self.out_norm = nn.LayerNorm(d)
        self.proj = nn.Linear(d, d, bias=False)
    def forward(self, x):
        for ln, blk in zip(self.norms, self.blocks):
            if self.mode == 'pre':
                x = x + blk(ln(x))
            else:  # post
                x = ln(x + blk(x))
        return self.proj(self.out_norm(x))

def train(mode, L, lr=1e-3):
    torch.manual_seed(0)
    m = Stack(L, d, mode)
    opt = torch.optim.Adam(m.parameters(), lr=lr)
    losses = []
    for _ in range(n_steps):
        pred = m(X)
        loss = ((pred - y) ** 2).mean()
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())
    return losses

fig, ax = plt.subplots(figsize=(7, 4))
for L in [4, 16]:
    for mode in ['pre', 'post']:
        losses = train(mode, L)
        ax.semilogy(losses, label=f'L={L} {mode}-norm')
ax.set_xlabel('step'); ax.set_ylabel('loss')
ax.set_title('Loss: pre-norm vs. post-norm at depth')
ax.legend(); plt.tight_layout(); plt.show()

**Sub-finding 4.** *Did post-norm at L=16 train at all? Was pre-norm visibly more stable? What was the loss ratio at step 100 between the two at L=16?*

## Experiment 5 — residual-stream norm growth (pre-norm only, at init)

In [ ]:
torch.manual_seed(0)
L = 16
m = Stack(L, d, 'pre')
x = torch.randn(8, d)   # batch of 8 tokens
norms = [x.norm(dim=-1).mean().item()]
with torch.no_grad():
    for ln, blk in zip(m.norms, m.blocks):
        x = x + blk(ln(x))
        norms.append(x.norm(dim=-1).mean().item())

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(norms, 'o-')
ax.set_xlabel('depth (block index)')
ax.set_ylabel('mean ||residual stream||')
ax.set_title('Residual-stream norm growth at init')
plt.tight_layout(); plt.show()
print(f'norm at L=0: {norms[0]:.3f}')
print(f'norm at L={L}: {norms[-1]:.3f}')
print(f'ratio: {norms[-1] / norms[0]:.2f}')

**Sub-finding 5.** *Did the norm grow monotonically? At what rate — linear in $L$, square-root, exponential?*

## Finding

*3–6 sentences. Did pre-norm dominate at depth as predicted? What was surprising about the residual stream growth — did it match the theoretical $O(\sqrt{L})$ that random-walk arguments would predict?*